In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import rankdata
import numpy as np
from collections import Counter
import os

In [2]:
class CBFEngine:
    def __init__(self, data_path='data/cbf_df.csv', ratings_count_path='data/movies_ratings_count.csv'):
        self.df_cbf = pd.read_csv(data_path)

        lista_compilado = self.df_cbf['geral'].tolist()

        cvec = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1, max_df=.85)
        sf = cvec.fit_transform(lista_compilado)

        transformer = TfidfTransformer()
        tfidf_matrix = transformer.fit_transform(sf)

        self.cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

        df_ratings_count = pd.read_csv(ratings_count_path)
        list_ratings_count = np.array(df_ratings_count.loc[:, 'count_ratings'].to_list())
        log_ratings = np.log1p(list_ratings_count)
        self.pop_score = (log_ratings - log_ratings.min()) / (log_ratings.max() - log_ratings.min())

    def recomendar_filmes_perfil(self, user_id, n_recomendacoes=50, df_traing_path='data_spliting/training.csv'):
        df_traing = pd.read_csv(df_traing_path)


        df_user = df_traing.loc[df_traing['userId'] == user_id]
        list_id_movies = df_user['movieId'].to_list()

        indices_usuario = self.df_cbf[self.df_cbf['movie_id'].isin(list_id_movies)].index.tolist()

        if not indices_usuario:
            return pd.DataFrame()

        n_filmes = self.cosine_sim.shape[0]
        score_acumulado = np.zeros(n_filmes)

        for idx in indices_usuario:
            sim_scores_filme = self.cosine_sim[idx]
            score_acumulado = np.maximum(score_acumulado, sim_scores_filme)

        sim_scores_total = score_acumulado

        sim_norm = rankdata(sim_scores_total) / len(sim_scores_total)
        pop_norm = rankdata(self.pop_score) / len(self.pop_score)

        peso_conteudo = 0.95
        peso_pop = 0.05
        score_hibrido = (sim_norm * peso_conteudo) + (pop_norm * peso_pop)

        sim_scores = list(enumerate(score_hibrido))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores_filtrados = [s for s in sim_scores if s[0] not in indices_usuario]

        top_sim_scores = sim_scores_filtrados[:n_recomendacoes]
        filme_indices = [i[0] for i in top_sim_scores]

        recomendacoes = self.df_cbf.iloc[filme_indices][['movie_id', 'title']].copy()
        recomendacoes['score_recomendacao'] = [i[1] for i in top_sim_scores]

        return recomendacoes

    def verifica_relevantes(self, user_id, recomendacoes, df_testing_path='data_spliting/testing.csv'):
        df_testing = pd.read_csv(df_testing_path)
        recomendacoes = recomendacoes.loc[:, 'movie_id'].tolist()
        filmes_user_testing = df_testing.loc[df_testing['userId'] == user_id]
        total = pd.Series(recomendacoes).isin(filmes_user_testing['movieId']).sum()
        print(f'O total de filmes para o {user_id} relevantes é de: {total}')
        return total

    def validar_modelo_completo(self, df_test, k=10, df_train_path='data_spliting/training.csv'):
        usuarios_no_teste = df_test['userId'].unique()
        df_train = pd.read_csv(df_train_path)

        total_precision = 0
        total_recall = 0
        contagem_usuarios = 0
        resultados_por_usuario = []

        for user_id in usuarios_no_teste:
            filmes_reais = set(df_test[df_test['userId'] == user_id]['movieId'])

            rec_df = self.recomendar_filmes_perfil(user_id, n_recomendacoes=k)

            if rec_df.empty:
                continue

            filmes_recomendados = set(rec_df['movie_id'])

            hits = len(filmes_recomendados.intersection(filmes_reais))

            precision_user = hits / k
            recall_user = hits / len(filmes_reais)

            total_precision += precision_user
            total_recall += recall_user
            contagem_usuarios += 1

            filmes_assistidos = df_train[df_train['userId'] == user_id]['movieId'].tolist()

            resultados_por_usuario.append({
                'user_id': user_id,
                'precision': precision_user,
                'recall': recall_user,
                'hits': hits,
                'filmes_recomendados': rec_df.copy(),
                'filmes_assistidos': filmes_assistidos,
                'filmes_teste': list(filmes_reais),
            })

        metricas_finais = {
            'Mean Precision@K': total_precision / contagem_usuarios,
            'Mean Recall@K': total_recall / contagem_usuarios,
            'Usuarios Avaliados': contagem_usuarios
        }

        return metricas_finais, resultados_por_usuario

    def exibir_top_usuarios_por_precisao(self, resultados_por_usuario, top_n=10):
        top_usuarios = sorted(resultados_por_usuario, key=lambda x: x['precision'], reverse=True)[:top_n]

        for rank, user_data in enumerate(top_usuarios, start=1):
            user_id               = user_data['user_id']
            precision             = user_data['precision']
            recall                = user_data['recall']
            hits                  = user_data['hits']
            rec_df                = user_data['filmes_recomendados']
            filmes_assistidos_ids = user_data['filmes_assistidos']
            filmes_teste_ids      = set(user_data['filmes_teste'])

            print(f"\n{'='*75}")
            print(f"  TOP {rank:>2} | Usuário ID: {user_id} | Precision@10: {precision:.1%} | Recall@10: {recall:.1%} | Hits: {hits}")
            print(f"{'='*75}")

            assistidos_df = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_assistidos_ids)][['movie_id', 'title']]
            print(f"\n  [FILMES ASSISTIDOS - TREINO]  ({len(assistidos_df)} filmes)")
            print(f"  {'-'*60}")
            for _, row in assistidos_df.iterrows():
                print(f"    {row['movie_id']:>7}  {row['title']}")

            teste_df = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_teste_ids)][['movie_id', 'title']]
            print(f"\n  [FILMES DO TESTE - GABARITO]  ({len(teste_df)} filmes)")
            print(f"  {'-'*60}")
            for _, row in teste_df.iterrows():
                print(f"    {row['movie_id']:>7}  {row['title']}")

            indices_assistidos = self.df_cbf[self.df_cbf['movie_id'].isin(filmes_assistidos_ids)].index.tolist()
            indices_rec        = self.df_cbf[self.df_cbf['movie_id'].isin(rec_df['movie_id'])].index.tolist()

            score_por_idx = dict(zip(indices_rec, rec_df['score_recomendacao'].tolist()))

            print(f"\n  [FILMES RECOMENDADOS + VETORES DE SIMILARIDADE]")
            print(f"  (similaridade cosseno em relação aos {len(indices_assistidos)} filmes do perfil)")
            print(f"  {'-'*75}")
            print(f"  {'#':>2}  {'Título':<50}  {'Score':>7}  {'Sim Média':>9}  {'Sim Máx':>8}  {'Hit?':>5}")
            print(f"  {'-'*75}")

            for pos, (idx_rec, (_, row)) in enumerate(zip(indices_rec, rec_df.iterrows()), start=1):
                if indices_assistidos:
                    sim_vec = self.cosine_sim[idx_rec][indices_assistidos]
                    sim_med = float(sim_vec.mean())
                    sim_max = float(sim_vec.max())
                else:
                    sim_med = sim_max = 0.0

                hit_marker = ' HIT' if row['movie_id'] in filmes_teste_ids else '    '
                titulo = str(row['title'])[:50]
                print(f"  {pos:>2}. {titulo:<50}  {row['score_recomendacao']:>7.4f}  {sim_med:>9.4f}  {sim_max:>8.4f}  {hit_marker}")

        print(f"\n{'='*75}")

    def gerar_estudo_cobertura(self, df_test, ks=[100, 200, 300],
                               output_dir='estudo_hibrido',
                               df_train_path='data_spliting/training.csv'):
        os.makedirs(output_dir, exist_ok=True)

        for k in ks:
            resultados = []

            for user_id in df_test['userId'].unique():
                filmes_reais = set(df_test[df_test['userId'] == user_id]['movieId'])

                rec_df = self.recomendar_filmes_perfil(user_id, n_recomendacoes=k, df_traing_path=df_train_path)

                if rec_df.empty:
                    continue

                filmes_recomendados = set(rec_df['movie_id'])

                apareceram     = list(filmes_recomendados.intersection(filmes_reais))
                nao_apareceram = list(filmes_reais - filmes_recomendados)
                porcentagem    = round(len(apareceram) / len(filmes_reais) * 100, 2)

                resultados.append({
                    'userId': user_id,
                    'porcentagem_coberta': porcentagem,
                    'filmes_apareceram': apareceram,
                    'filmes_nao_apareceram': nao_apareceram,
                })

            df_resultado = pd.DataFrame(resultados)
            caminho = os.path.join(output_dir, f'cobertura_cbf_k{k}.csv')
            df_resultado.to_csv(caminho, index=False)
            print(f"[K={k}] Salvo em '{caminho}' — {len(df_resultado)} usuários | Cobertura média: {df_resultado['porcentagem_coberta'].mean():.2f}%")

In [3]:
cbf = CBFEngine()

In [4]:
df_teste = pd.read_csv('data_spliting/testing.csv')
metricas, resultados_usuarios = cbf.validar_modelo_completo(df_teste)
print(metricas)

{'Mean Precision@K': 0.14016528925619826, 'Mean Recall@K': 0.07152382309267342, 'Usuarios Avaliados': 605}


In [5]:
df_teste = pd.read_csv('data_spliting/testing.csv')
# cbf.gerar_estudo_cobertura(df_teste, ks=[100, 200, 300, 500, 1000, 2000, 5000], output_dir='estudo_hibrido')